In [1]:
import pandas as pd
import json
import glob
# from tqdm.notebook import tqdm
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
from sklearn.metrics import precision_score, recall_score, f1_score
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
tqdm.pandas()


In [2]:
category_schemas = json.load(open('../config/LLM_SDoH_tag.json', 'r'))

In [3]:
sentence_list = []
tag_list = []

In [4]:
for data_path in tqdm(glob.glob('../data/splitted_data/fold_*/test.json')):
    data = json.load(open(data_path,'r'))
    for i in data:
        sentence_list.append(' '.join(i['tokens']))
        tags = list(set([x.split('-')[-1] for x in i['ner_tags'] if x != 'O']))
        if tags == []:
            tag_list.append(['None'])
        else:
            tag_list.append(tags)

100%|█████████████████████████████████████████████| 5/5 [00:05<00:00,  1.03s/it]


In [5]:
data = pd.DataFrame({'Sentence': sentence_list, 'True': tag_list})

In [6]:
system_prompt = open('../prompts/Trigger_word.txt', 'r').read()

In [7]:
client = OpenAI()
def get_tags(sentence):
    try:
        response = client.chat.completions.create(
            model="gpt-4o", 
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Classify this sentence: '{sentence}'"}
            ],
            tools=[{
                "type": "function",
                "function": category_schemas
            }],
            tool_choice={
                "type": "function", 
                "function": {"name": category_schemas["name"]}
            }
        )

        tool_call = response.choices[0].message.tool_calls[0]
        parsed_data = json.loads(tool_call.function.arguments)
        # Extract using the correct key from your schema
        return parsed_data.get("selected_categories", [])

    except Exception as e:
        # If an error occurs, print it and return an empty list to keep the process moving
        print(f"API Error: {e}")
        return []


In [8]:
# 1. Extract the sentences into a standard Python list
sentences_list = data['Sentence'].tolist()

# 2. Set up the ThreadPoolExecutor
# max_workers=10 means it will process 10 sentences at the exact same time. 
# You can increase this, but be careful of OpenAI API rate limits (requests per minute).
with ThreadPoolExecutor(max_workers=10) as executor:
    # 3. Map the function to the list of sentences and wrap with tqdm for the progress bar
    results = list(tqdm(executor.map(get_tags, sentences_list), total=len(sentences_list)))

100%|███████████████████████████████████████| 2961/2961 [03:03<00:00, 16.11it/s]


In [12]:
data['predict'] = results

In [13]:
data['True'] = data['True'].apply(lambda x: [] if x == ['None'] else x)

In [14]:
data.to_csv("../output/GPT-4o-trigger.csv")

In [15]:
def compute_metrics(true_labels_list, pred_labels_list):
    total_true_positives = 0
    total_predicted = 0
    total_actual = 0
    
    # Renamed loop variables to true_item and pred_item for clarity
    for true_item, pred_item in zip(true_labels_list, pred_labels_list):
        true_set = set(true_item)
        pred_set = set(pred_item)
        
        true_positives = len(true_set.intersection(pred_set))
        total_true_positives += true_positives
        total_predicted += len(pred_set)
        total_actual += len(true_set)
    
    precision = total_true_positives / total_predicted if total_predicted > 0 else 0.0
    recall = total_true_positives / total_actual if total_actual > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    
    return precision, recall, f1

In [16]:
y_true = data['True'].tolist()
y_pred = data['predict'].tolist()

precision, recall, f1 = compute_metrics(y_true, y_pred)

print(f"Micro-Averaged Precision: {precision:.4f}")
print(f"Micro-Averaged Recall: {recall:.4f}")
print(f"Micro-Averaged F-1 Score: {f1:.4f}")

Micro-Averaged Precision: 0.5057
Micro-Averaged Recall: 0.6862
Micro-Averaged F-1 Score: 0.5823


In [37]:
gpt4o_results = ["GPT-4o"]
gpt4o_results += ["N/A"] * 6
gpt4o_results += [precision, recall, f1]

In [38]:
model_result = pd.read_csv("../results/032726/model_overall_summary.csv", index_col=0)

In [39]:
# Convert the list into a new DataFrame with the same columns
new_row = pd.DataFrame([gpt4o_results], columns=model_result.columns)

# Concatenate them and ignore the old index to keep it continuous
model_result = round(pd.concat([model_result, new_row], ignore_index=True),3)

In [41]:
model_result.to_csv("../results/032726/model_overall_summary.csv")

In [42]:
model_result.to_excel("../results/032726/model_overall_summary.xlsx")

In [43]:
from collections import defaultdict

def compute_metrics_per_class(all_true_labels, all_pred_labels, all_classes):
    """
    all_true_labels: List[List[str]]  # true labels per instance
    all_pred_labels: List[List[str]]  # predicted labels per instance
    all_classes: List[str]            # all 17 possible label values
    """

    # Store counts per class
    tp = defaultdict(int)
    fp = defaultdict(int)
    fn = defaultdict(int)

    for true_labels, pred_labels in zip(all_true_labels, all_pred_labels):
        true_set = set(true_labels)
        pred_set = set(pred_labels)

        for label in all_classes:
            if label in true_set and label in pred_set:
                tp[label] += 1
            elif label not in true_set and label in pred_set:
                fp[label] += 1
            elif label in true_set and label not in pred_set:
                fn[label] += 1
            # true negative is not used for P/R/F1

    # Compute precision, recall, f1 per class
    results = {}

    for label in all_classes:
        precision = tp[label] / (tp[label] + fp[label]) if (tp[label] + fp[label]) > 0 else 0.0
        recall = tp[label] / (tp[label] + fn[label]) if (tp[label] + fn[label]) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

        results[label] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "tp": tp[label],
            "fp": fp[label],
            "fn": fn[label]
        }

    return results


In [44]:
all_classes = ['Adherence',
 'Concern',
 'Education',
 'Employment',
 'Financial',
 'Healthcare',
 'Insurance',
 'Literacy',
 'Living',
 'MentalHealth',
 'None',
 'Recommendation',
 'Smoke',
 'Social',
 'SubstanceUse',
 'Transportation',
 'Trauma']

In [45]:
all_classes = [
    'Adherence', 'Concern', 'Education', 'Employment', 'Financial', 
    'Healthcare', 'Insurance', 'Literacy', 'Living', 'MentalHealth', 
    'Recommendation', 'Smoke', 'Social', 'SubstanceUse', 'Transportation', 'Trauma'
]

# 2. Extract the lists from your dataframe
y_true = data['True'].tolist()
y_pred = data['predict'].tolist()

# 3. Run your evaluation function
category_results = compute_metrics_per_class(y_true, y_pred, all_classes)

# 4. Convert the dictionary output into a Pandas DataFrame for easy viewing
results_df = pd.DataFrame(category_results).T

# We can sort the values by F-1 score to see which classes perform the best
results_df = results_df.sort_values(by='f1', ascending=False)



In [46]:
results_df

,precision,recall,f1,tp,fp,fn
Smoke,0.875000,1.000000,0.933333,42.0,6.0,0.0
Employment,0.689474,0.922535,0.789157,131.0,59.0,11.0
SubstanceUse,0.629310,0.986486,0.768421,73.0,43.0,1.0
Education,0.669565,0.855556,0.751220,77.0,38.0,13.0
Living,0.557491,0.946746,0.701754,160.0,127.0,9.0
Financial,0.544073,0.774892,0.639286,179.0,150.0,52.0
MentalHealth,0.500000,0.777778,0.608696,91.0,91.0,26.0
Trauma,0.675926,0.536765,0.598361,73.0,35.0,63.0
Adherence,0.549383,0.605442,0.576052,89.0,73.0,58.0
Healthcare,0.463869,0.627760,0.533512,199.0,230.0,118.0


In [72]:
overall_results = ["overall"]
overall_results += [precision, recall, f1]
overall_results += ["N/A"] * 3
tmp_df = pd.DataFrame([overall_results[1:]], columns=results_df.columns, index = [overall_results[0]])

In [75]:
results_df = pd.concat([results_df, tmp_df])

In [81]:
model_result_dict = json.load(open("../results/032726/model_results.json","r"))

In [82]:
model_result_dict['GPT-4o'] = {"average":{}}

In [83]:
for i, content in results_df.iterrows():
    model_result_dict['GPT-4o']['average'][i] = {"sentence-level":{'precision': round(content['precision'],3),
                                                'recall': round(content['recall'],3),
                                                'f-1':round(content['f1'],3)}}

In [84]:
with open("../results/032726/model_results.json","w") as f:
    json.dump(model_result_dict, f)